# 04 — Build Performance Analysis


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('..').resolve()))

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns

from build_optimiser.config import Config
from build_optimiser.graph import (
    attach_metrics,
    critical_path,
    critical_path_length,
    load_graph,
    topological_depth,
    transitive_dependants,
)
from build_optimiser.simulation import replay_git_history, simulate_incremental_build

cfg = Config.from_yaml('../../config.yaml')
file_df = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)


## 1. Load Ownership Data

Read team ownership and contributor group assignments produced by notebook 02.


In [ ]:
ownership_df = pd.read_parquet(cfg.processed_data_dir / 'target_ownership.parquet')
groups_df = pd.read_parquet(cfg.processed_data_dir / 'contributor_groups.parquet')

# Derive owning_group_label from groups_df (group_id -> group_label mapping)
if 'owning_group_label' not in ownership_df.columns and 'owning_group_id' in ownership_df.columns:
    label_map = groups_df[['group_id', 'group_label']].drop_duplicates().set_index('group_id')['group_label']
    ownership_df['owning_group_label'] = ownership_df['owning_group_id'].map(label_map).fillna('unknown')

print('Ownership rows:', len(ownership_df))
print('Contributor group rows:', len(groups_df))
print()
print('Ownership columns:', ownership_df.columns.tolist())
print('Groups columns:', groups_df.columns.tolist())
print()
print('Teams:', ownership_df['owning_group_label'].value_counts().head(10).to_dict())

## 2. Critical Path Analysis

Compute the critical path through the build DAG weighted by `total_build_time_ms`. Classify each target as critical, near-critical (slack < 10 % of CP length), or other. Annotate with owning team and codegen fraction.


In [ ]:
# Compute critical path using total_build_time_ms as the weight attribute.
# critical_path() returns an ordered list of target names on the longest path.
weight_attr = 'total_build_time_ms'
cp_nodes = critical_path(G, weight_attr)
cp_len = critical_path_length(G, weight_attr)
cp_set = set(cp_nodes)

print(f'Critical path length: {cp_len / 1000:.1f} s')
print(f'Critical path steps:  {len(cp_nodes)}')
print()
print('Critical path targets:')
for t in cp_nodes:
    w = G.nodes[t].get(weight_attr) or 0
    print(f'  {t:50s}  {w/1000:6.1f} s')


In [ ]:
# Compute slack per target.
# Slack = critical_path_length - (earliest_start + own_weight + longest_suffix_through_target).
# Use a forward + backward pass on the reversed graph.

# Topological order: dependants before their dependencies (A->B means A first).
topo = list(nx.topological_sort(G))

# Forward pass: earliest_finish[n] = earliest time target n finishes.
earliest_finish = {}
for n in reversed(topo):  # reversed topo = dependencies first
    w = G.nodes[n].get(weight_attr) or 0
    deps = list(G.successors(n))
    earliest_start_n = max((earliest_finish[d] for d in deps), default=0)
    earliest_finish[n] = earliest_start_n + w

earliest_start = {}
for n in reversed(topo):
    deps = list(G.successors(n))
    earliest_start[n] = max((earliest_finish[d] for d in deps), default=0)

# Backward pass: latest_finish[n] = latest n can finish without delaying the build.
latest_finish = {}
for n in topo:  # dependants first
    w = G.nodes[n].get(weight_attr) or 0
    dependants = list(G.predecessors(n))
    if not dependants:
        latest_finish[n] = cp_len
    else:
        latest_finish[n] = min(
            (latest_finish[d] - (G.nodes[d].get(weight_attr) or 0) for d in dependants),
            default=cp_len,
        )

slack_records = []
for n in G.nodes():
    w = G.nodes[n].get(weight_attr) or 0
    ef = earliest_finish.get(n, 0)
    lf = latest_finish.get(n, cp_len)
    slack = lf - ef
    slack_records.append({'cmake_target': n, 'slack_ms': max(slack, 0), 'own_time_ms': w})

slack_df = pd.DataFrame(slack_records)
slack_df = slack_df.merge(
    ownership_df[['cmake_target', 'owning_group_label']].drop_duplicates(),
    on='cmake_target',
    how='left',
).merge(
    target_df[['cmake_target', 'codegen_ratio']].drop_duplicates(),
    on='cmake_target',
    how='left',
)

near_critical_threshold = cp_len * 0.10
slack_df['criticality'] = 'other'
slack_df.loc[slack_df['cmake_target'].isin(cp_set), 'criticality'] = 'critical'
slack_df.loc[
    (slack_df['criticality'] == 'other') & (slack_df['slack_ms'] < near_critical_threshold),
    'criticality',
] = 'near-critical'

print(slack_df['criticality'].value_counts())
print()
print('Near-critical targets (slack < 10 % of CP):')
near_crit = slack_df[slack_df['criticality'] == 'near-critical'].sort_values('slack_ms')
print(near_crit[['cmake_target', 'slack_ms', 'own_time_ms', 'owning_group_label']].to_string(index=False))


In [ ]:
# DAG visualisation with critical path highlighted.
# Restrict to targets reachable from or reaching a CP node to keep the plot legible.

# Build a subgraph: CP nodes + their direct predecessors and successors.
relevant_nodes = set(cp_nodes)
for n in cp_nodes:
    relevant_nodes.update(G.predecessors(n))
    relevant_nodes.update(G.successors(n))

# Limit to at most 120 nodes to keep the plot readable.
if len(relevant_nodes) > 120:
    # Keep CP nodes and the heaviest non-CP nodes.
    non_cp = sorted(
        relevant_nodes - cp_set,
        key=lambda n: G.nodes[n].get(weight_attr) or 0,
        reverse=True,
    )[: 120 - len(cp_nodes)]
    relevant_nodes = cp_set | set(non_cp)

sub = G.subgraph(relevant_nodes).copy()

# Hierarchical layout: x = topological depth, y = spread within level.
# Compute depth via DP on topological order (longest path from any root).
depth_map = {}
for n in reversed(list(nx.topological_sort(sub))):
    deps = list(sub.successors(n))
    depth_map[n] = max((depth_map.get(d, 0) + 1 for d in deps), default=0)

depth_groups: dict[int, list] = {}
for node, depth in depth_map.items():
    depth_groups.setdefault(depth, []).append(node)

pos = {}
for depth, members in depth_groups.items():
    for i, t in enumerate(members):
        pos[t] = (depth, -(i - len(members) / 2))

color_map = {
    'critical': '#d62728',
    'near-critical': '#ff7f0e',
    'other': '#aec7e8',
}
criticality_lookup = dict(zip(slack_df['cmake_target'], slack_df['criticality']))
node_colors = [color_map.get(criticality_lookup.get(n, 'other'), '#aec7e8') for n in sub.nodes()]

fig, ax = plt.subplots(figsize=(18, 10))
nx.draw_networkx(
    sub,
    pos=pos,
    ax=ax,
    node_color=node_colors,
    node_size=400,
    font_size=6,
    arrows=True,
    arrowsize=10,
    edge_color='#cccccc',
    with_labels=True,
)
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=color_map['critical'], label='Critical path'),
    Patch(facecolor=color_map['near-critical'], label='Near-critical (slack < 10 %)'),
    Patch(facecolor=color_map['other'], label='Other'),
]
ax.legend(handles=legend_elements, loc='upper left')
ax.set_title('Build DAG — critical path highlighted', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.show()

## 3. Rebuild Amplification Analysis

For each historical commit, identify the modified targets and their transitive dependants. The *amplification factor* is `|rebuild set| / |changed targets|`. High amplification indicates structural fragility.


In [ ]:
git_history = pd.read_csv(cfg.raw_data_dir / 'git_history_detail.csv')
print('Git history columns:', git_history.columns.tolist())
print('Total commits:', git_history['commit_hash'].nunique())
print(git_history.head(3))


In [ ]:
# Build a file -> target mapping from file_metrics.
file_to_target: dict[str, str] = {}
if 'source_file' in file_df.columns and 'cmake_target' in file_df.columns:
    for _, row in file_df[['source_file', 'cmake_target']].dropna().iterrows():
        file_to_target[row['source_file']] = row['cmake_target']

# Determine the column holding changed file paths.
file_col = next(
    (c for c in git_history.columns if 'file' in c.lower() or 'path' in c.lower()),
    git_history.columns[1] if len(git_history.columns) > 1 else None,
)
commit_col = next(
    (c for c in git_history.columns if 'commit' in c.lower() or 'hash' in c.lower()),
    git_history.columns[0],
)

print(f'Using commit column: {commit_col!r}, file column: {file_col!r}')

# Pre-build transitive dependant lookup to avoid repeated graph traversals.
td_cache: dict[str, set] = {}

def get_transitive_dependants(target: str) -> set[str]:
    if target not in td_cache:
        td_cache[target] = set(transitive_dependants(G, target))
    return td_cache[target]


amp_records = []
for commit_hash, group in git_history.groupby(commit_col):
    changed_files = group[file_col].dropna().tolist() if file_col else []
    changed_targets = {file_to_target[f] for f in changed_files if f in file_to_target}
    if not changed_targets:
        continue

    rebuild_set: set[str] = set(changed_targets)
    for t in changed_targets:
        if t in G:
            rebuild_set.update(get_transitive_dependants(t))

    amp_factor = len(rebuild_set) / len(changed_targets)
    amp_records.append({
        'commit_hash': commit_hash,
        'changed_targets': len(changed_targets),
        'rebuild_set_size': len(rebuild_set),
        'amplification_factor': amp_factor,
    })

amp_df = pd.DataFrame(amp_records)
print(f'Commits with mapped targets: {len(amp_df)}')
print(amp_df.describe())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of amplification factors.
axes[0].hist(amp_df['amplification_factor'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(amp_df['amplification_factor'].median(), color='red', linestyle='--', label=f'Median {amp_df["amplification_factor"].median():.1f}x')
axes[0].axvline(amp_df['amplification_factor'].quantile(0.90), color='orange', linestyle='--', label=f'P90 {amp_df["amplification_factor"].quantile(0.90):.1f}x')
axes[0].set_xlabel('Amplification factor')
axes[0].set_ylabel('Commit count')
axes[0].set_title('Rebuild amplification distribution')
axes[0].legend()

# Scatter: changed targets vs rebuild set size.
axes[1].scatter(
    amp_df['changed_targets'],
    amp_df['rebuild_set_size'],
    alpha=0.4,
    s=20,
    color='steelblue',
)
max_val = max(amp_df['rebuild_set_size'].max(), amp_df['changed_targets'].max())
axes[1].plot([0, max_val], [0, max_val], 'k--', alpha=0.4, label='1x amplification')
axes[1].set_xlabel('Changed targets per commit')
axes[1].set_ylabel('Rebuild set size')
axes[1].set_title('Changed targets vs rebuild set')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# Cross-team rebuild impact: for each commit, identify owning teams of changed targets
# vs teams in the rebuild set, to see how often changes trigger cross-team rebuilds.

target_to_team = dict(zip(ownership_df['cmake_target'], ownership_df['owning_group_label']))

cross_team_records = []
for commit_hash, group in git_history.groupby(commit_col):
    changed_files = group[file_col].dropna().tolist() if file_col else []
    changed_targets = {file_to_target[f] for f in changed_files if f in file_to_target}
    if not changed_targets:
        continue

    changed_teams = {target_to_team.get(t) for t in changed_targets} - {None}

    rebuild_set = set(changed_targets)
    for t in changed_targets:
        if t in G:
            rebuild_set.update(get_transitive_dependants(t))

    affected_teams = {target_to_team.get(t) for t in rebuild_set} - {None}
    cross_team_rebuilds = len(affected_teams - changed_teams)
    cross_team_records.append({
        'commit_hash': commit_hash,
        'changed_teams': len(changed_teams),
        'affected_teams': len(affected_teams),
        'cross_team_rebuilds': cross_team_rebuilds,
    })

cross_df = pd.DataFrame(cross_team_records)
print('Commits with cross-team rebuilds:', (cross_df['cross_team_rebuilds'] > 0).sum())
print('Fraction causing cross-team rebuilds:', f"{(cross_df['cross_team_rebuilds'] > 0).mean():.1%}")
print()
print(cross_df.describe())


## 4. Incremental Build Simulation

Replay git history through `simulate_incremental_build()` and `replay_git_history()` to estimate real per-commit build times under the current structure. Compare against a "perfect modularity" baseline where each target has zero transitive dependants.


In [ ]:
# Build target_times dict: cmake_target -> total_build_time_ms.
target_times: dict[str, float] = {
    row['cmake_target']: float(row.get('total_build_time_ms') or 0)
    for _, row in target_df.iterrows()
}

n_cores = getattr(cfg, 'n_cores', 8)  # fall back to 8 if not configured

# Prepare commits list for replay_git_history: list of sets of changed target names.
commits_as_targets = []
for commit_hash, group in git_history.groupby(commit_col):
    changed_files = group[file_col].dropna().tolist() if file_col else []
    changed_targets = {file_to_target[f] for f in changed_files if f in file_to_target}
    if changed_targets:
        commits_as_targets.append(changed_targets)

print(f'Commits with mapped targets: {len(commits_as_targets)}')
print(f'Using {n_cores} cores for simulation')


In [ ]:
# Replay git history to get per-commit simulated build times.
# replay_git_history expects a DataFrame with (commit_hash, source_file, ...) columns.
sim_results = replay_git_history(
    G,
    commits=git_history,
    file_to_target=file_to_target,
    target_times=target_times,
    n_cores=n_cores,
)

sim_ms = sim_results['build_time_ms'].values.astype(float)
sim_ms = sim_ms[sim_ms > 0]  # exclude zero-build commits
print(f'Commits with non-zero build: {len(sim_ms)}')
print(f'Mean build time:   {sim_ms.mean() / 1000:.1f} s')
print(f'Median build time: {np.median(sim_ms) / 1000:.1f} s')
print(f'P90 build time:    {np.percentile(sim_ms, 90) / 1000:.1f} s')
print(f'P99 build time:    {np.percentile(sim_ms, 99) / 1000:.1f} s')

In [ ]:
# Baseline: perfect modularity — each target compiles independently (no transitive deps).
# Simulated build = sum of own compile times of changed targets only, divided by n_cores.
baseline_ms = []
for changed_targets in commits_as_targets:
    own_times = [target_times.get(t, 0) for t in changed_targets]
    # With perfect parallelism: max(own_times) if serial critical path, or sum/n_cores.
    baseline_ms.append(sum(own_times) / n_cores)

baseline_arr = np.array(baseline_ms, dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sim_ms / 1000, bins=40, alpha=0.7, label='Actual graph', color='steelblue', edgecolor='white')
axes[0].hist(baseline_arr / 1000, bins=40, alpha=0.6, label='Perfect modularity baseline', color='green', edgecolor='white')
axes[0].axvline(np.median(sim_ms) / 1000, color='steelblue', linestyle='--')
axes[0].axvline(np.median(baseline_arr) / 1000, color='green', linestyle='--')
axes[0].set_xlabel('Simulated build time (s)')
axes[0].set_ylabel('Commit count')
axes[0].set_title('Incremental build time distribution')
axes[0].legend()

percentiles = [50, 75, 90, 95, 99]
sim_pct = [np.percentile(sim_ms, p) / 1000 for p in percentiles]
base_pct = [np.percentile(baseline_arr, p) / 1000 for p in percentiles]
x = np.arange(len(percentiles))
axes[1].bar(x - 0.2, sim_pct, 0.4, label='Actual graph', color='steelblue')
axes[1].bar(x + 0.2, base_pct, 0.4, label='Perfect modularity', color='green')
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'P{p}' for p in percentiles])
axes[1].set_ylabel('Build time (s)')
axes[1].set_title('Percentile comparison')
axes[1].legend()

plt.tight_layout()
plt.show()

median_overhead = (np.median(sim_ms) - np.median(baseline_arr)) / np.median(baseline_arr)
print(f'Median overhead vs perfect modularity: {median_overhead:.1%}')


## 5. Bottleneck Regression

Predict per-file compile time from structural features using a gradient boosting model. Permutation importances identify which features contribute most to slow compiles, and we compute counterfactual savings for each top factor.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Feature columns — use what's available.
candidate_features = [
    'code_lines',
    'preprocessed_bytes',
    'header_depth_max',
    'total_includes',
    'unique_headers',
    'object_size_bytes',
    'is_generated',
]
available_features = [c for c in candidate_features if c in file_df.columns]
target_col = next(
    (c for c in ['compile_time_ms', 'compile_time_total_ms'] if c in file_df.columns),
    None,
)
print('Available features:', available_features)
print('Target column:', target_col)


In [ ]:
if target_col and available_features:
    model_df = file_df[available_features + [target_col]].dropna()
    model_df = model_df[model_df[target_col] > 0]

    X = model_df[available_features].copy()
    # Ensure numeric — is_generated may be bool
    for col in X.columns:
        X[col] = pd.to_numeric(X[col], errors='coerce').fillna(0)
    y = np.log1p(model_df[target_col].values)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    gb = GradientBoostingRegressor(n_estimators=200, max_depth=4, learning_rate=0.05, random_state=42)
    gb.fit(X_train, y_train)

    r2 = gb.score(X_test, y_test)
    print(f'R² on test set: {r2:.3f}')

    perm_imp = permutation_importance(gb, X_test, y_test, n_repeats=10, random_state=42)
    imp_df = pd.DataFrame({
        'feature': available_features,
        'importance_mean': perm_imp.importances_mean,
        'importance_std': perm_imp.importances_std,
    }).sort_values('importance_mean', ascending=False)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(
        imp_df['feature'],
        imp_imp := imp_df['importance_mean'],
        xerr=imp_df['importance_std'],
        color='steelblue',
        capsize=3,
    )
    ax.set_xlabel('Permutation importance (decrease in R²)')
    ax.set_title('Feature importances for compile time prediction')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    # Counterfactual savings: reduce top feature by 25 % and estimate compile time reduction.
    print('\nCounterfactual savings (25 % reduction in each top feature):')
    X_test_arr = X_test.copy()
    baseline_pred = np.expm1(gb.predict(X_test_arr)).sum()
    for feat in imp_df.head(5)['feature']:
        X_cf = X_test_arr.copy()
        X_cf[feat] = X_cf[feat] * 0.75
        cf_pred = np.expm1(gb.predict(X_cf)).sum()
        saving_pct = (baseline_pred - cf_pred) / baseline_pred
        print(f'  {feat:30s}  {saving_pct:+.1%} total compile time')
else:
    print('Skipping regression: required columns not found in file_metrics.parquet')


## 6. Codegen Build Impact

Summarise code generators (targets with `codegen_ratio > 0`). Compare compile time per SLOC between generated and hand-written code.


In [ ]:
# Targets that have a codegen component.
codegen_targets = target_df[target_df.get('codegen_ratio', pd.Series(dtype=float)).fillna(0) > 0].copy()
if 'codegen_ratio' not in target_df.columns:
    codegen_targets = target_df[target_df.get('codegen_file_count', pd.Series(dtype=float)).fillna(0) > 0].copy()

print(f'Codegen targets: {len(codegen_targets)} / {len(target_df)}')
print()

summary_cols = [
    c for c in ['cmake_target', 'target_type', 'codegen_ratio', 'codegen_file_count',
                'compile_time_sum_ms', 'total_build_time_ms']
    if c in target_df.columns
]
print(codegen_targets[summary_cols].sort_values('compile_time_sum_ms', ascending=False).head(15).to_string(index=False))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: codegen fraction vs total compile time.
if 'codegen_ratio' in target_df.columns:
    axes[0].scatter(
        target_df['codegen_ratio'].fillna(0),
        target_df['compile_time_sum_ms'].fillna(0) / 1000,
        alpha=0.5,
        s=30,
        c=target_df['codegen_ratio'].fillna(0),
        cmap='YlOrRd',
    )
    axes[0].set_xlabel('Codegen fraction (0–1)')
    axes[0].set_ylabel('Total compile time (s)')
    axes[0].set_title('Codegen fraction vs compile time')

# Box plots: compile time per SLOC for generated vs hand-written.
sloc_col = next((c for c in ['code_lines', 'sloc', 'code_lines_total'] if c in target_df.columns), None)
if sloc_col and 'codegen_ratio' in target_df.columns:
    plot_df = target_df.copy()
    plot_df['is_generated'] = plot_df['codegen_ratio'].fillna(0) > 0.5
    plot_df['compile_ms_per_sloc'] = plot_df['compile_time_sum_ms'] / plot_df[sloc_col].replace(0, np.nan)
    plot_df = plot_df.dropna(subset=['compile_ms_per_sloc'])

    labels = {True: 'Generated', False: 'Hand-written'}
    groups = [plot_df[plot_df['is_generated'] == k]['compile_ms_per_sloc'].values for k in [False, True]]
    axes[1].boxplot(groups, labels=['Hand-written', 'Generated'], showfliers=False)
    axes[1].set_ylabel('Compile time per SLOC (ms)')
    axes[1].set_title('Compile efficiency: generated vs hand-written')
else:
    axes[1].text(0.5, 0.5, 'SLOC data not available', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
plt.show()
